https://pmc.ncbi.nlm.nih.gov/articles/PMC3147056/

https://cobrapy.readthedocs.io/en/latest/getting_started.html

# 0. Loading packages

In [ ]:
import cobra
from typing import Optional, Tuple, List
from cobra.io import load_model,read_sbml_model,write_sbml_model
from cobra.flux_analysis import flux_variability_analysis,single_reaction_deletion, pfba
from cobra.sampling import sample
import os
import pandas as pd
import numpy as np
import copy
import re
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm_notebook as tqdm
from pathlib import Path
import time

from itertools import islice
import datetime
import csv
import logging
import warnings
logging.getLogger("cobra.io.sbml").setLevel(logging.ERROR)
logging.getLogger("cobra").setLevel(logging.ERROR)
warnings.filterwarnings(    "ignore",    message=".*Series.__getitem__ treating keys as positions is deprecated.*")
from statsmodels.tools.sm_exceptions import ConvergenceWarning
import networkx as nx

%matplotlib inline
# If you want to increase the default width of the notebook
#HTML("<style>.container { width:100% !important; }</style>")
# The default height of the metabolic maps as displayed in the notebook
escherDefaultMapHeight = 800

# Suppress convergence warnings (expected for processes with a large number of genes)
warnings.simplefilter('ignore', ConvergenceWarning)

# Simple helper function that prints the current timestamp
def time_stamp():
    print(f"⏱️ Completed: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

%config InlineBackend.figure_format = 'retina'
time_stamp()

⏱️ 実行完了時刻: 2026-08-27 11:26:22


In [2]:
def take(n, iterable):
    """Return the first n items of the iterable as a list."""
    return list(islice(iterable, n))
time_stamp()

⏱️ 実行完了時刻: 2026-08-26 08:16:15


# 1. Load genome scale metabolic models constructed by Carveme

In [3]:
species=["ECOR"+str(n).zfill(2) for n in np.arange(1,73)]
models=["ECOR"+str(n).zfill(2)+"_model.xml" for n in np.arange(1,73)]
time_stamp()

⏱️ 実行完了時刻: 2026-08-26 08:16:15


In [4]:
# speceis names
species[0:3]

['ECOR01', 'ECOR02', 'ECOR03']

In [5]:
# model names
models[0:3]

['ECOR01_model.xml', 'ECOR02_model.xml', 'ECOR03_model.xml']

In [6]:
# model selections
model_sels=[11,24,32,69,71,72,31,42,50,48,38,36,51,66]
model_sels=[11,32,71,31,50,38,51]
model_sels = [x for x in range(1, 73) if x not in (29, 52)]
display(model_sels)

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72]

In [7]:
model=read_sbml_model("/Volumes/Versatile/ECOR/carveme/"+models[54])
display(model)
time_stamp()

Name,ECOR55_model
Memory address,1630cb4c0
Number of metabolites,1729
Number of reactions,2742
Number of genes,1668
Number of groups,0
Objective expression,1.0*Growth - 1.0*Growth_reverse_699ae
Compartments,"cytosol, periplasm, extracellular space"


⏱️ 実行完了時刻: 2026-08-26 08:16:18


In [8]:
display(model.solver)
time_stamp()

⏱️ 実行完了時刻: 2026-08-26 08:16:18


In [9]:
with warnings.catch_warnings():
    warnings.simplefilter(action='ignore', category=FutureWarning)
    my_md=model.medium
display(take(10,my_md.items()))
time_stamp()

[('EX_12ppd__R_e', 1000.0),
 ('EX_12ppd__S_e', 1000.0),
 ('EX_14glucan_e', 1000.0),
 ('EX_15dap_e', 1000.0),
 ('EX_23camp_e', 1000.0),
 ('EX_23ccmp_e', 1000.0),
 ('EX_23cgmp_e', 1000.0),
 ('EX_23cump_e', 1000.0),
 ('EX_23dappa_e', 1000.0),
 ('EX_25dkglcn_e', 1000.0)]

⏱️ 実行完了時刻: 2026-08-26 08:16:18


In [10]:
mtbrxn_list=[] # all metabolites in a gem
for reaction in model.reactions:      
    for metabolite, stoichiometry in reaction.metabolites.items():
        mtbrxn_item=pd.DataFrame([[reaction.id,metabolite.id,metabolite.name,metabolite.formula]],
                                 columns=["reaction","metabolite","name","formula"])
        mtbrxn_list.append(mtbrxn_item)    
mtbrxn_df=pd.concat(mtbrxn_list,axis=0)
mtbrxn_df=mtbrxn_df.reset_index(drop=True)
display(mtbrxn_df.head())
time_stamp()

,reaction,metabolite,name,formula
0,EX_12ppd__R_e,12ppd__R_e,"(R)-Propane-1,2-diol",C3H8O2
1,EX_12ppd__S_e,12ppd__S_e,"(S)-Propane-1,2-diol",C3H8O2
2,EX_14glucan_e,14glucan_e,"1,4-alpha-D-glucan",C36H62O31
3,EX_15dap_e,15dap_e,"1,5-Diaminopentane",C5H16N2
4,EX_23camp_e,23camp_e,"2',3'-Cyclic AMP",C10H11N5O6P


⏱️ 実行完了時刻: 2026-08-26 08:16:20


In [11]:
# example
display(mtbrxn_df[mtbrxn_df["reaction"]=="EX_ca2_e"])
time_stamp()


,reaction,metabolite,name,formula
83,EX_ca2_e,ca2_e,Calcium,Ca


⏱️ 実行完了時刻: 2026-08-26 08:16:20


In [12]:

#for i in tqdm(np.arange(len(models))):
#    model=read_sbml_model("/Volumes/Versatile/ECOR/carveme/"+models[i])
#    write_sbml_model(model,"/Volumes/Versatile/ECOR/carveme/"+models[i])

In [ ]:
# === Configuration: output directory (change as needed) ===
out_dir = Path("/Volumes/Versatile/ECOR/fba/output_rev10/model_exports")
out_dir.mkdir(parents=True, exist_ok=True)

def sanitize_for_filename(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z._-]+', '_', str(s))[:120]

def extract_objective_info(model):
    """
    The cobrapy Objective holds a linear expression.
    If expression.as_coefficients_dict() is available, it returns
    {Reaction (or Var): coeff}.
    """
    obj = model.objective
    direction = getattr(obj, "direction", None) or "max"

    obj_coeffs = {}
    expr = getattr(obj, "expression", None)
    if expr is not None:
        try:
            coeffs = expr.as_coefficients_dict()
            for k, v in coeffs.items():
                rid = getattr(k, "id", str(k))
                obj_coeffs[str(rid)] = float(v)
        except Exception:
            pass

    return obj_coeffs, ("max" if str(direction).lower().startswith("max") else "min")

def reaction_equation_safe(r, use_names=False):
    """
    Retrieve the reaction equation string from reaction object r.
    - First try r.reaction (property).
    - Fall back to build_reaction_string(use_metabolite_names=...) if unavailable.
    - As a last resort, assemble the equation manually from stoichiometry
      (using IDs or names).
    """
    # 1) Reaction.reaction property (works in most versions)
    try:
        eq = getattr(r, "reaction", None)
        if isinstance(eq, str) and eq.strip():
            return eq
    except Exception:
        pass

    # 2) build_reaction_string
    try:
        eq = r.build_reaction_string(use_metabolite_names=use_names)
        if isinstance(eq, str) and eq.strip():
            return eq
    except Exception:
        pass

    # 3) Manual assembly (last resort)
    try:
        mets = r.metabolites  # dict: {Metabolite: coefficient}
        left_terms = []
        right_terms = []
        for m, coeff in mets.items():
            name_or_id = m.name if (use_names and getattr(m, "name", None)) else m.id
            sto = abs(coeff)
            term = f"{sto:g} {name_or_id}" if not np.isclose(sto, 1.0) else f"{name_or_id}"
            if coeff < 0:
                left_terms.append(term)
            else:
                right_terms.append(term)
        left = " + ".join(left_terms) if left_terms else "∅"
        right = " + ".join(right_terms) if right_terms else "∅"
        arrow = "<=>" if r.lower_bound < 0 and r.upper_bound > 0 else "->"
        return f"{left} {arrow} {right}"
    except Exception:
        return ""

def export_model_genes_reactions(model, species_name: str, out_dir: Path):
    """
    Generated files:
      - {species}__genes.csv
          gene_id, gene_name, associated_reactions, reaction_count
      - {species}__reactions.csv
          reaction_id, reaction_name, subsystem, lower_bound, upper_bound,
          gene_reaction_rule, objective_coeff, is_in_objective,
          reaction_equation_ids, reaction_equation_names
      - {species}__objective_reactions.csv
          reaction_id, coeff, direction
    """
    sp_tag = sanitize_for_filename(species_name)

    # --- Extract objective ---
    obj_coeffs, obj_dir = extract_objective_info(model)
    obj_rxn_ids = set(obj_coeffs.keys())

    # --- genes ---
    gene_rows = []
    for g in model.genes:
        gname = getattr(g, "name", "")
        try:
            assoc_rxns = sorted({r.id for r in g.reactions})
        except Exception:
            assoc_rxns = []
        gene_rows.append({
            "gene_id": g.id,
            "gene_name": gname if gname is not None else "",
            "associated_reactions": ";".join(assoc_rxns),
            "reaction_count": len(assoc_rxns),
        })
    genes_df = (pd.DataFrame(gene_rows)
                  .sort_values(["gene_id"], kind="stable")
                  .reset_index(drop=True))

    # --- reactions ---
    rxn_rows = []
    for r in model.reactions:
        rname = getattr(r, "name", "")
        subsystem = getattr(r, "subsystem", "")
        lb = getattr(r, "lower_bound", np.nan)
        ub = getattr(r, "upper_bound", np.nan)
        gpr = getattr(r, "gene_reaction_rule", "")

        # Reaction equation (ID-based / name-based)
        eq_ids = reaction_equation_safe(r, use_names=False)
        eq_names = reaction_equation_safe(r, use_names=True)

        coeff = obj_coeffs.get(r.id, 0.0)
        rxn_rows.append({
            "reaction_id": r.id,
            "reaction_name": rname if rname is not None else "",
            "subsystem": subsystem if subsystem is not None else "",
            "lower_bound": float(lb) if lb is not None else np.nan,
            "upper_bound": float(ub) if ub is not None else np.nan,
            "gene_reaction_rule": gpr if gpr is not None else "",
            "objective_coeff": float(coeff),
            "is_in_objective": bool(r.id in obj_rxn_ids),
            "reaction_equation_ids": eq_ids,
            "reaction_equation_names": eq_names,
        })
    reactions_df = (pd.DataFrame(rxn_rows)
                      .sort_values(["is_in_objective", "reaction_id"], ascending=[False, True], kind="stable")
                      .reset_index(drop=True))

    # --- objective reactions list ---
    obj_rows = [{"reaction_id": rid, "coeff": float(c), "direction": obj_dir}
                for rid, c in obj_coeffs.items()]
    objective_df = (pd.DataFrame(obj_rows)
                      .sort_values(["reaction_id"], kind="stable")
                      .reset_index(drop=True))

    # --- save ---
    genes_path = out_dir / f"{sp_tag}__genes.csv"
    rxns_path = out_dir / f"{sp_tag}__reactions.csv"
    obj_path  = out_dir / f"{sp_tag}__objective_reactions.csv"

    genes_df.to_csv(genes_path, index=False)
    reactions_df.to_csv(rxns_path, index=False)
    objective_df.to_csv(obj_path, index=False)

    return genes_path, rxns_path, obj_path

# ========== Multiple species / multiple models ==========
# Assumes species, models, and model_sels are already defined
selected_indices = [m - 1 for m in model_sels]  # Convert 1-based to 0-based indices

for idx in tqdm(selected_indices, desc="Export models"):
    sp = species[idx]

    # Prepend the base path if model_path is not an absolute path (adjust as needed)
    model_path = models[idx]
    if not model_path.startswith("/"):
        model_path = "/Volumes/Versatile/ECOR/carveme/" + model_path

    model = read_sbml_model(model_path)

    gp, rp, op = export_model_genes_reactions(model, sp, out_dir)
    print(f"[{sp}] saved:\n  - {gp.name}\n  - {rp.name}\n  - {op.name}")
time_stamp()

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/3736381189.py:160: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for idx in tqdm(selected_indices, desc="Export models"):


Export models:   0%|          | 0/70 [00:00<?, ?it/s]

[ECOR01] saved:
  - ECOR01__genes.csv
  - ECOR01__reactions.csv
  - ECOR01__objective_reactions.csv
[ECOR02] saved:
  - ECOR02__genes.csv
  - ECOR02__reactions.csv
  - ECOR02__objective_reactions.csv
[ECOR03] saved:
  - ECOR03__genes.csv
  - ECOR03__reactions.csv
  - ECOR03__objective_reactions.csv
[ECOR04] saved:
  - ECOR04__genes.csv
  - ECOR04__reactions.csv
  - ECOR04__objective_reactions.csv
[ECOR05] saved:
  - ECOR05__genes.csv
  - ECOR05__reactions.csv
  - ECOR05__objective_reactions.csv
[ECOR06] saved:
  - ECOR06__genes.csv
  - ECOR06__reactions.csv
  - ECOR06__objective_reactions.csv
[ECOR07] saved:
  - ECOR07__genes.csv
  - ECOR07__reactions.csv
  - ECOR07__objective_reactions.csv
[ECOR08] saved:
  - ECOR08__genes.csv
  - ECOR08__reactions.csv
  - ECOR08__objective_reactions.csv
[ECOR09] saved:
  - ECOR09__genes.csv
  - ECOR09__reactions.csv
  - ECOR09__objective_reactions.csv
[ECOR10] saved:
  - ECOR10__genes.csv
  - ECOR10__reactions.csv
  - ECOR10__objective_reactions.csv


In [14]:
display(model.reactions.Growth.reaction)
time_stamp()

'0.000223 10fthf_c + 0.513689 ala__L_c + 0.000223 amet_c + 0.295792 arg__L_c + 0.241055 asn__L_c + 0.241055 asp__L_c + 54.124831 atp_c + 0.005205 ca2_c + 0.005205 cl_c + 0.000576 coa_c + 0.0001 cobalt2_c + 0.133508 ctp_c + 0.000709 cu2_c + 0.09158 cys__L_c + 0.026166 datp_c + 0.027017 dctp_c + 0.027017 dgtp_c + 0.026166 dttp_c + 0.000223 fad_c + 0.006715 fe2_c + 0.007808 fe3_c + 0.26316 gln__L_c + 0.26316 glu__L_c + 0.612638 gly_c + 0.215096 gtp_c + 48.601527 h2o_c + 0.094738 his__L_c + 0.290529 ile__L_c + 0.195193 k_c + 0.450531 leu__L_c + 0.343161 lys__L_c + 0.153686 met__L_c + 0.008675 mg2_c + 0.000223 mlthf_c + 0.000691 mn2_c + 0.0001 mql8_c + 0.001831 nad_c + 0.000447 nadp_c + 0.185265 phe__L_c + 0.221055 pro__L_c + 0.000223 pydx5p_c + 0.000223 ribflv_c + 0.215792 ser__L_c + 0.004338 so4_c + 0.000223 thf_c + 0.000223 thmpp_c + 0.253687 thr__L_c + 0.056843 trp__L_c + 0.137896 tyr__L_c + 0.1 uaagmda_c + 0.144104 utp_c + 0.423162 val__L_c + 0.000341 zn2_c --> 53.95 adp_c + 53.95 h_c 

⏱️ 実行完了時刻: 2026-08-26 08:19:53


In [15]:
# all reactions involving carbon metabolites
## It takes about 6 min to complete this cell. The output dataframe (mtbrxn_all_df) has been saved and can be reloaded.
mtbrxn_all_models=[]
for i in tqdm(np.arange(len(models))):
    model=read_sbml_model("/Volumes/Versatile/ECOR/carveme/"+models[i])
    mtbrxn_list=[] # all metabolites in gems
    for reaction in model.reactions:      
        for metabolite, stoichiometry in reaction.metabolites.items():
            mtbrxn_item=pd.DataFrame([[reaction.id,metabolite.id,metabolite.name,metabolite.formula]],
                                     columns=["reaction","metabolite","name","formula"])
            mtbrxn_list.append(mtbrxn_item)    
    mtbrxn_df=pd.concat(mtbrxn_list,axis=0)
    mtbrxn_df=mtbrxn_df.reset_index(drop=True)
    mtbrxn_all_models.append(mtbrxn_df)
mtbrxn_all_df=pd.concat(mtbrxn_all_models)
mtbrxn_all_df=mtbrxn_all_df.drop_duplicates(subset=["reaction","metabolite"])
mtbrxn_all_df=mtbrxn_all_df.reset_index(drop=True)
non_carbons=["Cl","Co","Cu","Ca","Cd"]
mtbrxn_all_df["carbon_atom"]="Absence"
mtbrxn_all_df.loc[mtbrxn_all_df["formula"].str.contains(r'^C'),["carbon_atom"]]="Presence"
mtbrxn_all_df.loc[mtbrxn_all_df["formula"].isin(non_carbons),["carbon_atom"]]="Absence" # all metabolites in gems
mtbrxn_all_df=mtbrxn_all_df[mtbrxn_all_df["carbon_atom"]=="Presence"]
mtbrxn_all_df=mtbrxn_all_df.reset_index(drop=True)
display(mtbrxn_all_df)
time_stamp()

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/2580710644.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i in tqdm(np.arange(len(models))):


  0%|          | 0/72 [00:00<?, ?it/s]

,reaction,metabolite,name,formula,carbon_atom
0,EX_12ppd__R_e,12ppd__R_e,"(R)-Propane-1,2-diol",C3H8O2,Presence
1,EX_12ppd__S_e,12ppd__S_e,"(S)-Propane-1,2-diol",C3H8O2,Presence
2,EX_14glucan_e,14glucan_e,"1,4-alpha-D-glucan",C36H62O31,Presence
3,EX_15dap_e,15dap_e,"1,5-Diaminopentane",C5H16N2,Presence
4,EX_23camp_e,23camp_e,"2',3'-Cyclic AMP",C10H11N5O6P,Presence
...,...,...,...,...,...
8975,GLNt,gln__L_c,L-Glutamine,C5H10N2O3,Presence
8976,SHSL3,ch4s_c,Methanethiol CH4S,CH4S,Presence
8977,SHSL3,suchms_c,O-Succinyl-L-homoserine,C8H12NO6,Presence
8978,SHSL3,met__L_c,L-Methionine,C5H11NO2S,Presence


⏱️ 実行完了時刻: 2026-08-26 08:25:55


In [16]:
# All metabolites
## It takes about 6 min to complete this cell. The output dataframe (mtb_df) has been saved and can be reloaded.
mtb_all_models=[]
for i in tqdm(np.arange(len(models))):
    model=read_sbml_model("/Volumes/Versatile/ECOR/carveme/"+models[i])
    mtb_list=[] # all metabolites in gems    
    for metabolite in model.metabolites: # all metabolites in gems
        mtb_item=pd.DataFrame([[metabolite.id,metabolite.name,metabolite.formula]],
                              columns=["metabolite","name","formula"])
        mtb_list.append(mtb_item)    
    mtb_df=pd.concat(mtb_list,axis=0)
    mtb_df=mtb_df.reset_index(drop=True)
    mtb_df["species"]=species[i]
    mtb_all_models.append(mtb_df)
mtb_df=pd.concat(mtb_all_models)
mtb_df=mtb_df.drop_duplicates(subset=["metabolite"])
mtb_df=mtb_df.reset_index(drop=True)
non_carbons=["Cl","Co","Cu","Ca","Cd"]
mtb_df["carbon_atom"]="Absence"
mtb_df.loc[mtb_df["formula"].str.contains(r'^C'),["carbon_atom"]]="Presence"
mtb_df.loc[mtb_df["formula"].isin(non_carbons),["carbon_atom"]]="Absence" # all metabolites in gems
mtb_df=mtb_df.drop(columns="species")
display(mtb_df)
time_stamp()

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/587973852.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i in tqdm(np.arange(len(models))):


  0%|          | 0/72 [00:00<?, ?it/s]

,metabolite,name,formula,carbon_atom
0,10fthf_c,10-Formyltetrahydrofolate,C20H21N7O7,Presence
1,12dgr120_c,"1,2-Diacyl-sn-glycerol (didodecanoyl, n-C12:0)",C27H52O5,Presence
2,12dgr140_c,"1,2-Diacyl-sn-glycerol (ditetradecanoyl, n-C14:0)",C31H60O5,Presence
3,12dgr140_p,"1,2-Diacyl-sn-glycerol (ditetradecanoyl, n-C14:0)",C31H60O5,Presence
4,12dgr141_c,"1,2-Diacyl-sn-glycerol (ditetradec-7-enoyl, n-...",C31H56O5,Presence
...,...,...,...,...
1969,R_3htd58coa_c,3 hydroxy 5Z 8Z tetradecedienyl CoA,C35H52N7O18P3S,Presence
1970,R_3htd5coa_c,3 hydroxy 5Z tetradecenyl CoA,C35H56N7O18P3S,Presence
1971,d5kg_c,2-Deoxy-5-keto-D-gluconic acid,C6H9O6,Presence
1972,d5kgp_c,2-Deoxy-5-keto-D-gluconic acid 6-phosphate,C6H8O9P,Presence


⏱️ 実行完了時刻: 2026-08-26 08:29:52


In [17]:
# save
# mtbrxn_all_df.to_csv("/Volumes/Versatile/ECOR/fba/output_rev/mtbrxn_all_df.csv",index=False)
# mtb_df.to_csv("/Volumes/Versatile/ECOR/fba/output_rev/mtb_all_df.csv",index=False)
time_stamp()

⏱️ 実行完了時刻: 2026-08-26 08:29:52


# Define M9 media

In [18]:
# M9 media
M9_singles=pd.read_csv("/Volumes/Versatile/ECOR/fba/M9_singles_lowox.tsv",sep="\t")
carbons=M9_singles[M9_singles["type"]=="Carbon"]["name"]
display(M9_singles)
time_stamp()

,reaction,metabolite,global_id,flux,keggId,formulae,descriptive_name,type,name,mw,...,lactic acid,Fumarate,Malic acid,Pyruvate,Aspartate,Glutamate,Threonine,Arginine,Histidine,Lysine
0,EX_succ_e,succ_e,EX_succ_e,10,C00042,C4H4O4,Succinate,Carbon,Succinate,116.07000,...,0,0,0,0,0,0,0,0,0,0
1,EX_ala__L_e,ala__L_e,EX_ala__L_e,10,C00041,C3H7NO2,L-Alanine,Carbon,Alanine,89.09000,...,0,0,0,0,0,0,0,0,0,0
2,EX_pro__L_e,pro__L_e,EX_pro__L_e,10,C00148,C5H9NO2,L-Proline,Carbon,Proline,115.13000,...,0,0,0,0,0,0,0,0,0,0
3,EX_ac_e,ac_e,EX_ac_e,10,C00033,C2H3O2,Acetate,Carbon,Acetate,59.04000,...,0,0,0,0,0,0,0,0,0,0
4,EX_mnl_e,mnl_e,EX_mnl_e,10,C00392,C6H14O6,D-Mannitol,Carbon,Mannitol,182.17000,...,0,0,0,0,0,0,0,0,0,0
5,EX_cit_e,cit_e,EX_cit_e,10,C00158,C6H5O7,Citrate,Carbon,Citrate,189.12400,...,0,0,0,0,0,0,0,0,0,0
6,EX_glc__D_e,glc__D_e,EX_glc__D_e,10,C00031,C6H12O6,D-Glucose,Carbon,Glucose,180.15600,...,0,0,0,0,0,0,0,0,0,0
7,EX_rib__D_e,rib__D_e,EX_rib__D_e,10,C00121,C5H10O5,D-Ribose,Carbon,Ribose,150.13000,...,0,0,0,0,0,0,0,0,0,0
8,EX_glyc_e,glyc_e,EX_glyc_e,10,C00116,C3H8O3,Glycerol,Carbon,Glycerol,92.09382,...,0,0,0,0,0,0,0,0,0,0
9,EX_ser__L_e,ser__L_e,EX_ser__L_e,10,C00065,C3H7NO3,L-Serine,Carbon,Serine,105.09000,...,0,0,0,0,0,0,0,0,0,0


⏱️ 実行完了時刻: 2026-08-26 08:29:52


In [19]:
# carbon sources
display(carbons[0:3])
time_stamp()

0    Succinate
1      Alanine
2      Proline
Name: name, dtype: object

⏱️ 実行完了時刻: 2026-08-26 08:29:52


In [20]:
M9_media=copy.deepcopy(M9_singles)
M9_media[carbons]=M9_media[carbons].astype(float)
display(M9_media.head())
time_stamp()

,reaction,metabolite,global_id,flux,keggId,formulae,descriptive_name,type,name,mw,...,lactic acid,Fumarate,Malic acid,Pyruvate,Aspartate,Glutamate,Threonine,Arginine,Histidine,Lysine
0,EX_succ_e,succ_e,EX_succ_e,10,C00042,C4H4O4,Succinate,Carbon,Succinate,116.07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,EX_ala__L_e,ala__L_e,EX_ala__L_e,10,C00041,C3H7NO2,L-Alanine,Carbon,Alanine,89.09,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,EX_pro__L_e,pro__L_e,EX_pro__L_e,10,C00148,C5H9NO2,L-Proline,Carbon,Proline,115.13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,EX_ac_e,ac_e,EX_ac_e,10,C00033,C2H3O2,Acetate,Carbon,Acetate,59.04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,EX_mnl_e,mnl_e,EX_mnl_e,10,C00392,C6H14O6,D-Mannitol,Carbon,Mannitol,182.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


⏱️ 実行完了時刻: 2026-08-26 08:29:52


# Run FBA and extract essential reactions for optimal growth

<div class="alert-success">
<font size="6">the ECOR models (70 strains)</font> 
</div>

In [ ]:
# ----------------------------
# Parameters
# ----------------------------
model_sels= [x for x in range(1, 73) if x not in (29, 52)]

threshold_factor = 0.999    # Require KO_Z < threshold_factor * Z
eps = 1e-9
C_TOTAL = 120.0  # mmol C gDW⁻¹ h⁻¹ (following Leiby & Marx 2014)
C_TOTAL = 60.0  # mmol C gDW⁻¹ h⁻¹ (equivalent to 10 mmol glucose gDW⁻¹ h⁻¹)

def _count_carbons_from_formula(formula):
    """Count the number of carbon atoms (nC) in a molecular formula string
    (excluding Cl, Co, etc.)."""
    if not formula or not isinstance(formula, str):
        return 0
    # Match 'C' not followed by a lowercase letter (excludes Cl, etc.);
    # absent digit treated as 1
    matches = re.findall(r'C(?![a-z])(\d*)', formula)
    if not matches:
        return 0
    nC = 0
    for m in matches:
        nC += int(m) if (m and m.isdigit()) else 1
    return nC

def apply_carbon_normalized_medium_from_M9singles(model, carbon_name, M9_singles, aerobic=True, C_TOTAL=120.0):
    """
    Build model.medium from the 'carbon_name' column in M9_singles (your TSV file).
    - For Base rows, use the column value (typically 1000) as the uptake bound.
    - Among Carbon rows, only the row where name == carbon_name is
      carbon-normalised: bound = C_TOTAL / nC.
    - model.medium expects positive uptake bounds; always set positive values.

    Returns: (nC_used, ex_id)
    """
    # Initialise all medium bounds to 0, keeping the current medium keys as a template
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        medium = model.medium.copy()
    for k in list(medium.keys()):
        medium[k] = 0.0

    # Retrieve the column for this carbon source
    if carbon_name not in M9_singles.columns:
        raise KeyError(f"'{carbon_name}' not found in M9_singles.")

    col = M9_singles[carbon_name]

    # Set column values (positive uptake bounds) for all EX reactions present
    # in the model, regardless of type (Base or Carbon)
    for ex, val in zip(M9_singles["global_id"], col):
        if model.reactions.has_id(ex):
            v = float(val) if pd.notnull(val) else 0.0
            # This is a positive uptake bound — must not be negative
            medium[ex] = max(0.0, v)

    # Overwrite with carbon-normalised bounds only for EX reactions of the
    # target carbon source (type == 'Carbon' & name == carbon_name)
    rows = M9_singles[(M9_singles["type"]=="Carbon") & (M9_singles["name"]==carbon_name)]
    nC_used, ex_id_used = 0, None
    for ex, formula in zip(rows["global_id"], rows["formulae"]):
        if model.reactions.has_id(ex):
            nC = _count_carbons_from_formula(formula)
            if nC > 0:
                medium[ex] = C_TOTAL / nC   # Positive uptake bound (lower bound is set internally by cobrapy)
                nC_used = nC
                ex_id_used = ex

    # Toggle O2 availability (TSV has 1000 as the Base value, which is usually
    # fine as-is; revised to 1 in a later version)
    if model.reactions.has_id("EX_o2_e"):
        if not aerobic:
            medium["EX_o2_e"] = 0.0  # Anaerobic
        # Aerobic (default): use the TSV value (typically 1000); revised to 10 in a later version


    # Apply to model
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.medium = medium

    return nC_used, ex_id_used
time_stamp()

⏱️ 実行完了時刻: 2026-08-26 14:56:28


## Test using ECOR11

In [ ]:
# Example: test ECOR11 × Succinate
idx = [m-1 for m in [11]][0]
sp = species[idx]
model = read_sbml_model("/Volumes/Versatile/ECOR/carveme/" + models[idx])

carbon0 = "Succinate"  # must match the column name in M9_singles
with model:
    nC_used, ex_id = apply_carbon_normalized_medium_from_M9singles(
        model, carbon0, M9_singles, aerobic=True, C_TOTAL=60.0
    )
    sol = model.optimize()
    print("species:", sp, "| carbon:", carbon0, "| ex:", ex_id, "| nC:", nC_used)
    print("status:", sol.status, "| Z:", sol.objective_value)
    # Verify a subset of medium bounds (check that they are positive uptake bounds)
    for k in [ex_id, "EX_nh4_e", "EX_pi_e", "EX_so4_e", "EX_o2_e", "EX_h2o_e"]:
        if k in model.medium:
            print(k, "=", model.medium[k])
time_stamp()

species: ECOR11 | carbon: Succinate | ex: EX_succ_e | nC: 4
status: optimal | Z: 0.39483221597931417
EX_succ_e = 15.0
EX_nh4_e = 1000.0
EX_pi_e = 1000.0
EX_so4_e = 1000.0
EX_o2_e = 10.0
EX_h2o_e = 1000.0
⏱️ 実行完了時刻: 2026-08-26 14:57:23


## Main loop for the ECOR strains

In [ ]:
# Action for ECOR strains
model_sels=[11,32,71,31,50,38,55] # ECOR11, ECOR32, ECOR71, ECOR31, ECOR50, ECOR38, ECOR55
model_sels= [x for x in range(1, 73) if x not in (29, 52)]

threshold_factor = 0.999    # Require KO_Z < threshold_factor * Z (essential if growth drops by more than 0.1%)
eps = 1e-9
C_TOTAL = 120.0  # mmol C gDW⁻¹ h⁻¹ (following Leiby & Marx 2014)
C_TOTAL = 60.0  # mmol C gDW⁻¹ h⁻¹ (equivalent to 10 mmol glucose gDW⁻¹ h⁻¹)

SAVE_ALL_KO = False        # Whether to save full KO results CSV (False recommended for large outputs)
SAVE_ESSENTIAL_ONLY = True
SAVE_SUMMARY = True

def get_growth_status(model):
    sol = model.optimize()
    status = getattr(sol, "status", "unknown")
    Z = float(sol.objective_value) if (sol is not None and sol.status == "optimal" and sol.objective_value is not None) else np.nan
    return Z, status

def sanitize_for_filename(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z._-]+', '_', str(s))[:120]

# Output directory
out_dir = Path("/Volumes/Versatile/ECOR/fba/output_rev10/essential_by_species")
out_dir.mkdir(parents=True, exist_ok=True)

# Carbon source columns to iterate over (fixed in advance since they are the same each time)
carbon_cols = [c for c in M9_singles.columns if c not in
               ["reaction","metabolite","global_id","flux","keggId","formulae","descriptive_name","type","name","mw"]]

selected_indices = [m-1 for m in model_sels]

# --- Utility to normalise column names and IDs in SRD results ---
def _normalize_srd_result(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)
    # Handle versions where the index holds reaction IDs
    if df.index.name is not None or isinstance(df.index, pd.MultiIndex):
        df = df.reset_index()
    # Normalise varying column names (growth/objective, ids/reaction/reactions/index)
    if "growth" in df.columns:
        df = df.rename(columns={"growth": "KO_Z"})
    elif "objective" in df.columns:
        df = df.rename(columns={"objective": "KO_Z"})
    # Rename the reaction ID column to reaction_id
    for cand in ["ids", "reaction", "reactions", "index"]:
        if cand in df.columns and "reaction_id" not in df.columns:
            df = df.rename(columns={cand: "reaction_id"})
    if "reaction_id" not in df.columns:
        raise KeyError(f"Cannot identify a reaction_id column in single_reaction_deletion result: {list(df.columns)}")

    # Ensure reaction_id is always a str (handles set/tuple/list values)
    def _norm_rid(x):
        if isinstance(x, (set, list, tuple)):
            if len(x) == 1:
                return str(next(iter(x)))
            return ";".join(sorted(map(str, x)))  # Multiple items: join with ';' (adjust as needed)
        return str(x)
    df["reaction_id"] = df["reaction_id"].apply(_norm_rid)

    # Cast KO_Z to float; treat NaN as 0.0
    # (conservative: ambiguous result treated as lethal — change if needed)
    df["KO_Z"] = pd.to_numeric(df["KO_Z"], errors="coerce").fillna(0.0)

    # Normalise the status column name
    if "status" in df.columns:
        df = df.rename(columns={"status": "status_KO"})
    else:
        df["status_KO"] = "unknown"

    return df[["reaction_id", "KO_Z", "status_KO"]].copy()

for idx in tqdm(selected_indices, desc="Models"):
    sp = species[idx]
    model_path = "/Volumes/Versatile/ECOR/carveme/" + models[idx]
    model = read_sbml_model(model_path)

    rxn_ids = [r.id for r in model.reactions]
    rxn_names = {r.id: (r.name if hasattr(r, "name") else "") for r in model.reactions}

    # Accumulate results per species
    records = []

    for carbon in tqdm(carbon_cols, desc=f"{sp}", leave=False):
        with model:
            nC_used, ex_id = apply_carbon_normalized_medium_from_M9singles(
                model, carbon, M9_singles, aerobic=True, C_TOTAL=C_TOTAL
            )
            Z, status_base = get_growth_status(model)

            # If baseline growth is unavailable, record one row and move to the next carbon source
            if not np.isfinite(Z) or Z <= 0:
                records.append(dict(
                    species=sp, carbon=carbon, ex_id=ex_id, nC=nC_used,
                    baseline_Z=np.nan,
                    reaction_id="(NA)", reaction_name="",
                    KO_Z=np.nan, drop_frac=np.nan,
                    essential=False, status_baseline=status_base, status_KO="NA"
                ))
                continue

            thr = threshold_factor * Z

            # ==== Key for performance: batch single-reaction deletions with SRD (processes=1 is fastest) ====
            srd_res = single_reaction_deletion(model, rxn_list=rxn_ids, processes=1, method="fba")
            srd_res = _normalize_srd_result(srd_res)

            # Attach additional metadata
            srd_res["reaction_name"] = srd_res["reaction_id"].map(rxn_names).fillna("")
            srd_res["species"]       = sp
            srd_res["carbon"]        = carbon
            srd_res["ex_id"]         = ex_id
            srd_res["nC"]            = nC_used
            srd_res["baseline_Z"]    = Z
            srd_res["drop_frac"]     = np.where(Z > 0, (Z - srd_res["KO_Z"]) / Z, np.nan)
            srd_res["essential"]     = srd_res["KO_Z"] < thr
            srd_res["status_baseline"] = status_base

            # Append to records
            records.extend(srd_res.to_dict(orient="records"))

    # === Save results per species ===
    sp_tag = sanitize_for_filename(sp)
    base = f"{sp_tag}"

    result_df = pd.DataFrame.from_records(records)

    if SAVE_ALL_KO:
        all_path = out_dir / f"{base}__all_ko.csv"
        result_df.to_csv(all_path, index=False)

    if SAVE_ESSENTIAL_ONLY:
        essential_df = result_df[result_df["essential"]].copy()
        ess_path = out_dir / f"{base}__essential_only.csv"
        essential_df.to_csv(ess_path, index=False)
        display(essential_df.head())

    if SAVE_SUMMARY:
        # Number of essential reactions per carbon source
        essential_df = result_df[result_df["essential"]].copy()
        summary_df = (essential_df
                      .groupby(["species","carbon"])["reaction_id"]
                      .nunique()
                      .reset_index(name="n_essential_reactions"))
        sum_path = out_dir / f"{base}__summary_by_carbon.csv"
        summary_df.to_csv(sum_path, index=False)
        display(summary_df.head())
time_stamp()

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/2989189673.py:72: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for idx in tqdm(selected_indices, desc="Models"):


Models:   0%|          | 0/70 [00:00<?, ?it/s]

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/2989189673.py:83: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for carbon in tqdm(carbon_cols, desc=f"{sp}", leave=False):


ECOR01:   0%|          | 0/34 [00:00<?, ?it/s]

,reaction_id,KO_Z,status_KO,reaction_name,species,carbon,ex_id,nC,baseline_Z,drop_frac,essential,status_baseline
10,DPCOAK,0.0,optimal,Dephospho-CoA kinase,ECOR01,Succinate,EX_succ_e,4,0.397643,1.0,True,optimal
26,SUCBZS,0.0,optimal,O-succinylbenzoate-CoA synthase,ECOR01,Succinate,EX_succ_e,4,0.397643,1.0,True,optimal
31,CLtex,0.0,optimal,Chloride (Cl-1) transport via diffusion (extra...,ECOR01,Succinate,EX_succ_e,4,0.397643,1.0,True,optimal
43,CDPMEK,0.0,optimal,4-(cytidine 5'-diphospho)-2-C-methyl-D-erythri...,ECOR01,Succinate,EX_succ_e,4,0.397643,1.0,True,optimal
48,PMDPHT,0.0,optimal,Pyrimidine phosphatase,ECOR01,Succinate,EX_succ_e,4,0.397643,1.0,True,optimal


,species,carbon,n_essential_reactions
0,ECOR01,Acetate,238
1,ECOR01,Alanine,240
2,ECOR01,Arabinose,241
3,ECOR01,Arginine,236
4,ECOR01,Aspartate,236


... (273 outputs left out)

ECOR72:   0%|          | 0/34 [00:00<?, ?it/s]

,reaction_id,KO_Z,status_KO,reaction_name,species,carbon,ex_id,nC,baseline_Z,drop_frac,essential,status_baseline
11,DPCOAK,0.0,optimal,Dephospho-CoA kinase,ECOR72,Succinate,EX_succ_e,4,0.394832,1.0,True,optimal
28,SUCBZS,0.0,optimal,O-succinylbenzoate-CoA synthase,ECOR72,Succinate,EX_succ_e,4,0.394832,1.0,True,optimal
33,CLtex,0.0,optimal,Chloride (Cl-1) transport via diffusion (extra...,ECOR72,Succinate,EX_succ_e,4,0.394832,1.0,True,optimal
45,CDPMEK,0.0,optimal,4-(cytidine 5'-diphospho)-2-C-methyl-D-erythri...,ECOR72,Succinate,EX_succ_e,4,0.394832,1.0,True,optimal
50,PMDPHT,0.0,optimal,Pyrimidine phosphatase,ECOR72,Succinate,EX_succ_e,4,0.394832,1.0,True,optimal


,species,carbon,n_essential_reactions
0,ECOR72,Acetate,238
1,ECOR72,Alanine,240
2,ECOR72,Arabinose,240
3,ECOR72,Arginine,233
4,ECOR72,Aspartate,236


⏱️ 実行完了時刻: 2026-08-27 04:43:13


## pFBA for the ECOR strains 

In [ ]:
# --- Configuration parameters ---
model_sels = [x for x in range(1, 73) if x not in (29, 52)]
selected_indices = [m - 1 for m in model_sels]

C_TOTAL = 120.0  # mmol C gDW^-1 h^-1
C_TOTAL = 60.0  # mmol C gDW⁻¹ h⁻¹ (equivalent to 10 mmol glucose gDW⁻¹ h⁻¹)
eps = 1e-6       # Threshold for treating numerically negligible fluxes as zero

# Output directory
out_dir = Path("/Volumes/Versatile/ECOR/fba/output_rev10/flux_distribution_by_species")
out_dir.mkdir(parents=True, exist_ok=True)

def sanitize_for_filename(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z._-]+', '_', str(s))[:120]

# Extract carbon source columns
carbon_cols = [c for c in M9_singles.columns if c not in
               ["reaction", "metabolite", "global_id", "flux", "keggId", "formulae", "descriptive_name", "type", "name", "mw"]]

# Loop over strains (species)
for idx in tqdm(selected_indices, desc="Models"):
    sp = species[idx]
    model_path = f"/Volumes/Versatile/ECOR/carveme/{models[idx]}"
    model = read_sbml_model(model_path)

    # Dictionary holding flux results across all carbon sources for one strain
    # Keys: reaction_id; values: {carbon_source: flux_value}
    flux_dict = {r.id: {} for r in model.reactions}
    growth_records = []

    for carbon in tqdm(carbon_cols, desc=f"{sp}", leave=False):
        with model:
            # Apply medium conditions (carbon source)
            nC_used, ex_id = apply_carbon_normalized_medium_from_M9singles(
                model, carbon, M9_singles, aerobic=True, C_TOTAL=C_TOTAL
            )
            
            try:
                # Run pFBA (maximise growth + minimise total flux)
                sol = pfba(model)
                status = sol.status
                
                if status == "optimal" and sol.objective_value is not None and sol.objective_value > 1e-4:
                    Z = float(sol.objective_value)
                    fluxes = sol.fluxes
                else:
                    Z = np.nan
                    fluxes = None
            except Exception:
                status = "infeasible/error"
                Z = np.nan
                fluxes = None

            # Record growth result
            growth_records.append({
                "species": sp,
                "carbon": carbon,
                "ex_id": ex_id,
                "nC": nC_used,
                "baseline_Z": Z,
                "status": status
            })

            # Store flux values for each reaction (NaN if growth is not feasible)
            for r in model.reactions:
                if fluxes is not None:
                    val = float(fluxes[r.id])
                    # Round numerically negligible fluxes to 0.0
                    flux_dict[r.id][carbon] = 0.0 if abs(val) < eps else val
                else:
                    flux_dict[r.id][carbon] = np.nan

    # === Save results ===
    sp_tag = sanitize_for_filename(sp)
    
    # 1. Build the flux matrix data frame
    #    (rows: reaction_id, columns: carbon source)
    flux_df = pd.DataFrame.from_dict(flux_dict, orient="index")
    flux_df.index.name = "reaction_id"
    
    # Reorder columns and save
    flux_df = flux_df[carbon_cols]
    flux_csv_path = out_dir / f"{sp_tag}__flux_matrix.csv"
    flux_df.to_csv(flux_csv_path)

    # 2. Save per-medium growth rate and status summary
    growth_df = pd.DataFrame(growth_records)
    growth_csv_path = out_dir / f"{sp_tag}__growth_summary.csv"
    growth_df.to_csv(growth_csv_path, index=False)
    
    display(growth_df.head())
    display(flux_df.head())
time_stamp()

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/2328669540.py:21: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for idx in tqdm(selected_indices, desc="Models"):


Models:   0%|          | 0/70 [00:00<?, ?it/s]

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/2328669540.py:31: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for carbon in tqdm(carbon_cols, desc=f"{sp}", leave=False):


ECOR01:   0%|          | 0/34 [00:00<?, ?it/s]

,species,carbon,ex_id,nC,baseline_Z,status
0,ECOR01,Succinate,EX_succ_e,4,321.041293,optimal
1,ECOR01,Alanine,EX_ala__L_e,3,426.367851,optimal
2,ECOR01,Proline,EX_pro__L_e,5,265.399445,optimal
3,ECOR01,Acetate,EX_ac_e,2,338.990896,optimal
4,ECOR01,Mannitol,EX_mnl_e,6,505.479778,optimal


,Succinate,Alanine,Proline,Acetate,Mannitol,Citrate,Glucose,Ribose,Glycerol,Serine,...,lactic acid,Fumarate,Malic acid,Pyruvate,Aspartate,Glutamate,Threonine,Arginine,Histidine,Lysine
reaction_id,,,,,,,,,,,,,,,,,,,,,
EX_12ppd__R_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
EX_12ppd__S_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
EX_14glucan_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
EX_15dap_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
EX_23camp_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0


... (273 outputs left out)

ECOR72:   0%|          | 0/34 [00:00<?, ?it/s]

,species,carbon,ex_id,nC,baseline_Z,status
0,ECOR72,Succinate,EX_succ_e,4,320.567125,optimal
1,ECOR72,Alanine,EX_ala__L_e,3,426.829693,optimal
2,ECOR72,Proline,EX_pro__L_e,5,264.814811,optimal
3,ECOR72,Acetate,EX_ac_e,2,338.611998,optimal
4,ECOR72,Mannitol,EX_mnl_e,6,508.028081,optimal


,Succinate,Alanine,Proline,Acetate,Mannitol,Citrate,Glucose,Ribose,Glycerol,Serine,...,lactic acid,Fumarate,Malic acid,Pyruvate,Aspartate,Glutamate,Threonine,Arginine,Histidine,Lysine
reaction_id,,,,,,,,,,,,,,,,,,,,,
EX_12ppd__R_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
EX_12ppd__S_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
EX_14glucan_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
EX_15dap_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
EX_23camp_e,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0


⏱️ 実行完了時刻: 2026-08-26 15:59:38


In [ ]:
# check

In [ ]:
# --- Configuration parameters ---
model_sels = [x for x in range(1, 73) if x not in (29, 52)]
selected_indices = [m - 1 for m in model_sels]

# Output directory
out_dir = Path("/Volumes/Versatile/ECOR/fba/output_rev10")
out_dir.mkdir(parents=True, exist_ok=True)

check_results = []

for idx in tqdm(selected_indices, desc="Checking Models"):
    sp = species[idx]
    model_fname = models[idx]
    model_path = f"/Volumes/Versatile/ECOR/carveme/{model_fname}"
    
    rec = {
        "index": idx + 1,
        "species": sp,
        "filename": model_fname,
        "n_reactions": 0,
        "n_metabolites": 0,
        "n_genes": 0,
        "default_growth_rate": None,
        "unbalanced_reactions": 0,  # Number of reactions with element/charge imbalance
        "atp_unbounded": None,
        "error": None
    }
    
    try:
        # Load model
        model = cobra.io.read_sbml_model(model_path)
        rec["n_reactions"] = len(model.reactions)
        rec["n_metabolites"] = len(model.metabolites)
        rec["n_genes"] = len(model.genes)
        
        # 1. Optimise under default medium conditions
        sol = model.optimize()
        rec["default_growth_rate"] = sol.objective_value if sol.status == "optimal" else 0.0
        
        # 2. Element and charge conservation check (mass/charge balance check)
        # Directly check each reaction for formula inconsistencies
        # (violations of mass and charge conservation)
        unbalanced = 0
        for rxn in model.reactions:
            # Check internal reactions only (excluding boundary reactions
            # such as Exchange, Demand, and Sink)
            if not rxn.boundary:
                check_dict = rxn.check_mass_balance()
                if len(check_dict) > 0:
                    unbalanced += 1
        rec["unbalanced_reactions"] = unbalanced
        
        # 3. Check for energy-generating cycles (unbounded ATP production)
        with model:
            # Block all boundary reactions (Exchange, etc.) to prevent any nutrient uptake
            for rxn in model.boundary:
                rxn.lower_bound = max(0, rxn.lower_bound)
            
            # Detect ATP maintenance/hydrolysis reactions (ATPM, DM_atp_c_, etc.)
            atp_candidates = [r for r in model.reactions if "ATPM" in r.id or "DM_atp" in r.id or "RC00015" in r.id]
            if atp_candidates:
                model.objective = atp_candidates[0]
                atp_sol = model.optimize()
                # True if ATP flux exceeds the threshold (1e-6) without any external nutrient supply
                rec["atp_unbounded"] = (atp_sol.status == "optimal" and atp_sol.objective_value > 1e-6)
            else:
                rec["atp_unbounded"] = "ATPM_not_found"
                
    except Exception as e:
        rec["error"] = str(e)
        
    check_results.append(rec)

# Save and display
summary_df = pd.DataFrame(check_results)
summary_csv_path = out_dir / "model_quality_check_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

display(summary_df.head(10))

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/1971046188.py:11: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for idx in tqdm(selected_indices, desc="Checking Models"):


Checking Models:   0%|          | 0/70 [00:00<?, ?it/s]

,index,species,filename,n_reactions,n_metabolites,n_genes,default_growth_rate,unbalanced_reactions,atp_unbounded,error
0,1,ECOR01,ECOR01_model.xml,2561,1609,1607,65.999823,378,False,None
1,2,ECOR02,ECOR02_model.xml,2726,1719,1648,86.614675,403,False,None
2,3,ECOR03,ECOR03_model.xml,2634,1673,1644,65.784188,390,False,None
3,4,ECOR04,ECOR04_model.xml,2642,1673,1618,66.001498,395,False,None
4,5,ECOR05,ECOR05_model.xml,2595,1615,1623,86.614375,388,False,None
5,6,ECOR06,ECOR06_model.xml,2590,1622,1594,65.999932,388,False,None
6,7,ECOR07,ECOR07_model.xml,2609,1626,1667,82.501464,394,False,None
7,8,ECOR08,ECOR08_model.xml,2558,1600,1625,82.500051,383,False,None
8,9,ECOR09,ECOR09_model.xml,2581,1615,1604,65.999823,375,False,None
9,10,ECOR10,ECOR10_model.xml,2685,1704,1632,66.000041,401,False,None


<div class="alert-danger">
<font size="6">the iML1515 model (the K12 MG1655 strain)</font>
</div>

In [ ]:
species_name = "iML1515"
model_path = "/Volumes/Versatile/ECOR/mg1655/iML1515.xml"
out_dir = Path("/Volumes/Versatile/ECOR/mg1655/output_rev10/model_exports")
out_dir.mkdir(parents=True, exist_ok=True)

# Ensure the helper functions defined in the cell above
# (sanitize_for_filename, extract_objective_info, reaction_equation_safe,
#  export_model_genes_reactions) are available — either redefine them here
# or run that cell first.


model = read_sbml_model(model_path)
export_model_genes_reactions(model, species_name, out_dir)
print("Exported CSVs to:", out_dir)
time_stamp()

Exported CSVs to: /Volumes/Versatile/ECOR/mg1655/output_rev10/model_exports
⏱️ 実行完了時刻: 2026-08-27 10:44:53


## Measure processing speed

In [ ]:
# Test speed
threshold_factor = 0.999    # KO_Z < threshold_factor * Z を必須とする
eps = 1e-9
C_TOTAL = 120.0  # mmol C gDW⁻¹ h⁻¹ (following Leiby & Marx 2014)
C_TOTAL = 60.0  # mmol C gDW⁻¹ h⁻¹ (equivalent to 10 mmol glucose gDW⁻¹ h⁻¹)
model_path = "/Volumes/Versatile/ECOR/mg1655/iML1515.xml"
model = read_sbml_model(model_path)
carbon_cols = [c for c in M9_singles.columns if c not in
               ["reaction","metabolite","global_id","flux","keggId","formulae","descriptive_name","type","name","mw"]]
rxn_ids = [r.id for r in model.reactions]

def time_seq_loop(model, rxn_ids, Z, thr):
    t0 = time.time()
    n_essential = 0
    for rid in rxn_ids:
        KO_Z, _ = KO_growth_once(model, rid)
        # Treat KO_Z as 0 if nan/None (or skip — adjust as needed)
        if KO_Z is None or not np.isfinite(KO_Z):
            KO_Z = 0.0
        if KO_Z < thr:
            n_essential += 1
    return time.time() - t0, n_essential

def time_srd(model, rxn_ids, Z, thr, processes=1):
    t0 = time.time()
    with model:
        res = single_reaction_deletion(model, rxn_list=rxn_ids, processes=processes, method="fba")
    # Normalise column names
    if "growth" in res.columns:
        res = res.rename(columns={"growth": "KO_Z"})
    elif "objective" in res.columns:
        res = res.rename(columns={"objective":"KO_Z"})
    # Cast KO_Z to a comparable numeric type
    KO_Z = res["KO_Z"].astype(float).fillna(0.0).to_numpy()
    n_essential = int((KO_Z < thr).sum())
    return time.time() - t0, n_essential

# ---- Set medium and compute baseline growth ----
with model:
    nC_used, ex_id = apply_carbon_normalized_medium_from_M9singles(
        model, carbon_cols[0], M9_singles, aerobic=True, C_TOTAL=C_TOTAL
    )
    Z, status_base = get_growth_status(model)
    thr = (threshold_factor * Z) if (Z is not None and np.isfinite(Z) and Z > 0) else 0.0

# ---- Benchmark three methods ----
t_seq, n1 = time_seq_loop(model, rxn_ids, Z, thr)
t_srd1, n2 = time_srd(model, rxn_ids, Z, thr, processes=1)
t_srd4, n3 = time_srd(model, rxn_ids, Z, thr, processes=4)  # Adjust to match your environment

print(f"Seq loop (KO_growth_once x N): {t_seq:.2f}s, essential counted={n1}")
print(f"SRD processes=1             : {t_srd1:.2f}s, essential counted={n2}")
print(f"SRD processes=4             : {t_srd4:.2f}s, essential counted={n3}")

print(f"SRD processes=4             : {t_srd4:.2f}s, essential counted={n3}")
time_stamp()

/Users/tsuru/miniconda3/envs/fba/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


NameError: name 'KO_growth_once' is not defined

## Main

In [ ]:

# Switch output directory to mg1655
out_dir = Path("/Volumes/Versatile/ECOR/mg1655/output_rev10/essential_by_species")
out_dir.mkdir(parents=True, exist_ok=True)

# Single-model configuration
sp = "iML1515"  # species name (also used in filenames)
model_path = "/Volumes/Versatile/ECOR/mg1655/iML1515.xml"
model = read_sbml_model(model_path)

# Carbon source columns to iterate over (fixed in advance since they are the same each time)
carbon_cols = [c for c in M9_singles.columns if c not in
               ["reaction","metabolite","global_id","flux","keggId","formulae","descriptive_name","type","name","mw"]]

# Extract reaction IDs from the model
rxn_ids = [r.id for r in model.reactions]
rxn_names = {r.id: (r.name if hasattr(r, "name") else "") for r in model.reactions}

# Accumulate results per species
records = []

for carbon in tqdm(carbon_cols, desc=f"{sp}", leave=False):
    with model:
        nC_used, ex_id = apply_carbon_normalized_medium_from_M9singles(
            model, carbon, M9_singles, aerobic=True, C_TOTAL=60
        )
        Z, status_base = get_growth_status(model)

        # If baseline growth is unavailable, record one row and move to the next carbon source
        if not np.isfinite(Z):
            records.append(dict(
                species=sp, carbon=carbon, ex_id=ex_id, nC=nC_used,
                baseline_Z=np.nan,
                reaction_id="(NA)", reaction_name="",
                KO_Z=np.nan, drop_frac=np.nan,
                essential=False, status_baseline=status_base, status_KO="NA"
            ))
            continue

        thr = threshold_factor * Z  # e.g., 0.999 * Z
        
        # Single-reaction deletions
        srd_res = single_reaction_deletion(model, rxn_list=rxn_ids, processes=1, method="fba")
    # Normalise column names
    if "growth" in srd_res.columns:
        srd_res = srd_res.rename(columns={"growth":"KO_Z"})
    elif "objective" in srd_res.columns:
        srd_res = srd_res.rename(columns={"objective":"KO_Z"})

    srd_res = srd_res.reset_index()  # Reset index in case it holds reaction IDs
    # Rename the reaction ID column
    if "ids" in srd_res.columns:
        srd_res = srd_res.rename(columns={"ids":"reaction_id"})
    elif "index" in srd_res.columns and "reaction_id" not in srd_res.columns:
        srd_res = srd_res.rename(columns={"index":"reaction_id"})

    # Cast KO_Z to a comparable numeric type
    srd_res["KO_Z"] = srd_res["KO_Z"].astype(float)
    srd_res["KO_Z"] = srd_res["KO_Z"].fillna(0.0)   # Adjust as needed

    # Attach additional metadata
    
    # 1) Normalise the reaction ID column to reaction_id
    if not isinstance(srd_res, pd.DataFrame):
        srd_res = pd.DataFrame(srd_res)

     # Reset index in case it holds reaction IDs
    if srd_res.index.name is not None or isinstance(srd_res.index, pd.MultiIndex):
        srd_res = srd_res.reset_index()

    # Rename common candidate column names to reaction_id
    for cand in ["ids", "reaction", "reactions", "index"]:
        if cand in srd_res.columns and "reaction_id" not in srd_res.columns:
            srd_res = srd_res.rename(columns={cand: "reaction_id"})

    # Confirm the column exists; raise an error early if not
    if "reaction_id" not in srd_res.columns:
        raise KeyError("Cannot identify a reaction_id column in single_reaction_deletion result. "
                       "srd_res.columns = {}".format(list(srd_res.columns)))

    # 2) Normalise reaction_id values to str (handles set/tuple/list)
    def _norm_reaction_id(x):
        # Convert set/list/tuple to str
        if isinstance(x, (set, list, tuple)):
            if len(x) == 1:
                return str(next(iter(x)))
            # Multiple items: join with ';' (adjust as needed)
            return ";".join(sorted(map(str, x)))
        return str(x)

    srd_res["reaction_id"] = srd_res["reaction_id"].apply(_norm_reaction_id)
# --- End of normalisation block ---
    
    
    srd_res["reaction_name"] = srd_res["reaction_id"].map(rxn_names).fillna("")
    srd_res["species"]       = sp
    srd_res["carbon"]        = carbon
    srd_res["ex_id"]         = ex_id
    srd_res["nC"]            = nC_used
    srd_res["baseline_Z"]    = Z
    srd_res["drop_frac"]     = np.where(Z > 0, (Z - srd_res["KO_Z"]) / Z, np.nan)
    srd_res["essential"]     = srd_res["KO_Z"] < thr
    srd_res["status_baseline"] = status_base
    if "status" in srd_res.columns:
        srd_res = srd_res.rename(columns={"status":"status_KO"})
    else:
        srd_res["status_KO"] = "unknown"

    # Append to records (or write directly to CSV)
    records.extend(srd_res.to_dict(orient="records"))

# === Save results per species ===
sp_tag = sanitize_for_filename(sp)
base = f"{sp_tag}"  # No index suffix since this is a single model

result_df = pd.DataFrame.from_records(records)

if SAVE_ALL_KO:
    all_path = out_dir / f"{base}__all_ko.csv"
    result_df.to_csv(all_path, index=False)

if SAVE_ESSENTIAL_ONLY:
    essential_df = result_df[result_df["essential"]].copy()
    ess_path = out_dir / f"{base}__essential_only.csv"
    essential_df.to_csv(ess_path, index=False)
    display(essential_df.head())

if SAVE_SUMMARY:
    essential_df = result_df[result_df["essential"]].copy()
    summary_df = (essential_df
                  .groupby(["species","carbon"])["reaction_id"]
                  .nunique()
                  .reset_index(name="n_essential_reactions"))
    sum_path = out_dir / f"{base}__summary_by_carbon.csv"
    summary_df.to_csv(sum_path, index=False)
    display(summary_df.head())
time_stamp()

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/233577650.py:21: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for carbon in tqdm(carbon_cols, desc=f"{sp}", leave=False):


iML1515:   0%|          | 0/34 [00:00<?, ?it/s]

/Users/tsuru/miniconda3/envs/fba/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/Users/tsuru/miniconda3/envs/fba/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/Users/tsuru/miniconda3/envs/fba/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


,index,reaction_id,KO_Z,status_KO,reaction_name,species,carbon,ex_id,nC,baseline_Z,drop_frac,essential,status_baseline
6,6.0,DPCOAK,0.000000,optimal,Dephospho-CoA kinase,iML1515,Succinate,EX_succ_e,4,0.299682,1.000000,True,optimal
8,8.0,G3PD2,0.299305,optimal,Glycerol-3-phosphate dehydrogenase (NADP),iML1515,Succinate,EX_succ_e,4,0.299682,0.001256,True,optimal
9,9.0,ME2,0.283975,optimal,Malic enzyme (NADP),iML1515,Succinate,EX_succ_e,4,0.299682,0.052412,True,optimal
26,26.0,UPPDC1,0.000000,optimal,Uroporphyrinogen decarboxylase (uroporphyrinog...,iML1515,Succinate,EX_succ_e,4,0.299682,1.000000,True,optimal
27,27.0,THZPSN3,0.000000,optimal,Thiazole phosphate synthesis,iML1515,Succinate,EX_succ_e,4,0.299682,1.000000,True,optimal


,species,carbon,n_essential_reactions
0,iML1515,Acetate,355
1,iML1515,Alanine,361
2,iML1515,Arabinose,361
3,iML1515,Arginine,354
4,iML1515,Aspartate,354


⏱️ 実行完了時刻: 2026-08-27 10:58:55


## pFBA

In [ ]:
# --- Configuration parameters ---
C_TOTAL = 120.0  # mmol C gDW⁻¹ h⁻¹
C_TOTAL = 60.0   # mmol C gDW⁻¹ h⁻¹ (equivalent to 10 mmol glucose gDW⁻¹ h⁻¹)
eps = 1e-6       # Threshold for treating numerically negligible fluxes as zero

# Switch output directory to mg1655
out_dir = Path("/Volumes/Versatile/ECOR/mg1655/output_rev10/flux_distribution_by_species")
out_dir.mkdir(parents=True, exist_ok=True)

# Single-model configuration
sp = "iML1515"  # species name (also used in filenames)
model_path = "/Volumes/Versatile/ECOR/mg1655/iML1515.xml"
model = read_sbml_model(model_path)

def sanitize_for_filename(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z._-]+', '_', str(s))[:120]

# Extract carbon source columns to iterate over
carbon_cols = [c for c in M9_singles.columns if c not in
               ["reaction", "metabolite", "global_id", "flux", "keggId", "formulae", "descriptive_name", "type", "name", "mw"]]

# Dictionary holding flux results: {reaction_id: {carbon_source: flux_value}}
flux_dict = {r.id: {} for r in model.reactions}
growth_records = []

for carbon in tqdm(carbon_cols, desc=f"{sp}", leave=False):
    with model:
        # Apply medium conditions (carbon source)
        nC_used, ex_id = apply_carbon_normalized_medium_from_M9singles(
            model, carbon, M9_singles, aerobic=True, C_TOTAL=C_TOTAL
        )
        
        try:
            # Run pFBA (maximise growth + minimise total flux)
            sol = pfba(model)
            status = sol.status
            
            if status == "optimal" and sol.objective_value is not None and sol.objective_value > 1e-4:
                Z = float(sol.objective_value)
                fluxes = sol.fluxes
            else:
                Z = np.nan
                fluxes = None
        except Exception:
            status = "infeasible/error"
            Z = np.nan
            fluxes = None

        # Record growth result
        growth_records.append({
            "species": sp,
            "carbon": carbon,
            "ex_id": ex_id,
            "nC": nC_used,
            "baseline_Z": Z,
            "status": status
        })

        # Store flux values for each reaction (NaN if growth is not feasible)
        for r in model.reactions:
            if fluxes is not None:
                val = float(fluxes[r.id])
                # Round numerically negligible fluxes to 0.0
                flux_dict[r.id][carbon] = 0.0 if abs(val) < eps else val
            else:
                flux_dict[r.id][carbon] = np.nan

# === Save results ===
sp_tag = sanitize_for_filename(sp)

# 1. Build the flux matrix data frame (rows: reaction_id, columns: carbon source)
flux_df = pd.DataFrame.from_dict(flux_dict, orient="index")
flux_df.index.name = "reaction_id"

# Reorder columns and save
flux_df = flux_df[carbon_cols]
flux_csv_path = out_dir / f"{sp_tag}__flux_matrix.csv"
flux_df.to_csv(flux_csv_path)

# 2. Save growth rate and status summary
growth_df = pd.DataFrame(growth_records)
growth_csv_path = out_dir / f"{sp_tag}__growth_summary.csv"
growth_df.to_csv(growth_csv_path, index=False)

display(growth_df.head())
display(flux_df.head())
time_stamp()

/var/folders/7w/hjhkk8cx3bl57my8v2n8s8nh0000gn/T/ipykernel_19961/883303938.py:26: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for carbon in tqdm(carbon_cols, desc=f"{sp}", leave=False):


iML1515:   0%|          | 0/34 [00:00<?, ?it/s]

,species,carbon,ex_id,nC,baseline_Z,status
0,iML1515,Succinate,EX_succ_e,4,338.359258,optimal
1,iML1515,Alanine,EX_ala__L_e,3,480.000139,optimal
2,iML1515,Proline,EX_pro__L_e,5,342.027709,optimal
3,iML1515,Acetate,EX_ac_e,2,323.919787,optimal
4,iML1515,Mannitol,EX_mnl_e,6,563.994823,optimal


,Succinate,Alanine,Proline,Acetate,Mannitol,Citrate,Glucose,Ribose,Glycerol,Serine,...,lactic acid,Fumarate,Malic acid,Pyruvate,Aspartate,Glutamate,Threonine,Arginine,Histidine,Lysine
reaction_id,,,,,,,,,,,,,,,,,,,,,
CYTDK2,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,NaN,NaN
XPPT,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,NaN,NaN
HXPRT,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,NaN,NaN
NDPK5,0.000000,0.00960,0.000000,0.005073,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,NaN,NaN
SHK3Dr,0.114214,0.13543,0.100102,0.071564,0.192705,0.154109,0.202574,0.164074,0.186028,0.141203,...,0.089284,0.164063,0.164063,0.136697,0.164063,0.145365,0.15181,0.109841,NaN,NaN


⏱️ 実行完了時刻: 2026-08-26 15:00:43


In [ ]:
# Number of reactions to Pyr

In [ ]:
# Switch output directory to mg1655
out_dir = Path("/Volumes/Versatile/ECOR/mg1655/output_rev10/pathway_distance")
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Load the iML1515 model
# (download from the BIGG database or load from a local file)
sp = "iML1515"  # species name (also used in filenames)
model_path = "/Volumes/Versatile/ECOR/mg1655/iML1515.xml"
model = read_sbml_model(model_path)

# 1. List of cofactors, shared metabolites, and inorganic compounds to exclude
EXCLUDE_METABOLITES = {
    # Energy carriers, cofactors, and inorganic compounds
    "atp_c", "adp_c", "amp_c", "pi_c", "ppi_c", 
    "nad_c", "nadh_c", "nadp_c", "nadph_c", 
    "h2o_c", "h_c", "h_p", "co2_c", "o2_c", "coa_c",
    "nh4_c", "so4_c", "cl_c", "na1_c", "k_c", "ca2_c", "mg2_c",
    # Electron carriers and quinones
    "q8_c", "q8h2_c", "mqn8_c", "mqnh8_c", "2dmqn8_c", "2dmqnh8_c",
    "fad_c", "fadh2_c", "fe2_c", "fe3_c",
    # C2/C3 metabolites and conjugates that cause carbon shortcuts
    "accoa_c", "micit_c", "actp_c"
}

# 2. List of bypass reaction IDs to knock out (exclude)
# Block GCD (glucose dehydrogenase) and other non-primary periplasmic oxidation reactions
EXCLUDE_REACTIONS = {
    "GLCDpp",  # D-glucose dehydrogenase (periplasmic): blocks glc__D_p -> glcn_p
    "UGD",     # UDP-glucose dehydrogenase
}

G = nx.DiGraph()

for reaction in model.reactions:
    # Skip specified non-canonical bypass reactions from graph construction
    if reaction.id in EXCLUDE_REACTIONS:
        continue

    # Check whether this is a PTS reaction
    is_pts = "pts" in reaction.id.lower() or (
        "pep_c" in [m.id for m in reaction.reactants] and "pyr_c" in [m.id for m in reaction.products]
    )

    reactants = [m.id for m in reaction.reactants if m.id not in EXCLUDE_METABOLITES]
    products = [m.id for m in reaction.products if m.id not in EXCLUDE_METABOLITES]

    # For PTS reactions, remove only pyr_c from the products
    # (retain carbon flow via phosphorylation to g6p_c, etc.)
    if is_pts:
        products = [p for p in products if p != "pyr_c"]

    # Add forward edges (upper_bound > 0)
    if reaction.upper_bound > 0:
        for r in reactants:
            for p in products:
                G.add_edge(r, p, reaction=reaction.id)
                
    # Add reverse edges for reversible reactions (lower_bound < 0)
    if reaction.lower_bound < 0:
        for p in products:
            for r in reactants:
                G.add_edge(p, r, reaction=reaction.id)

# Carbon source list
carbon_sources = {
    "Succinate": "succ_e", "Alanine": "ala__L_e", "Proline": "pro__L_e",
    "Acetate": "ac_e", "Mannitol": "mnl_e", "Citrate": "cit_e",
    "Glucose": "glc__D_e", "Ribose": "rib__D_e", "Glycerol": "glyc_e",
    "Serine": "ser__L_e", "Sorbitol": "sbt__D_e", "Maltose": "malt_e",
    "Galactose": "gal_e", "Mannose": "man_e", "Trehalose": "tre_e",
    "Lactose": "lcts_e", "Melibiose": "melib_e", "Fructose": "fru_e",
    "Arabinose": "arab__L_e", "Rhamnose": "rmn_e", "Xylose": "xyl__D_e",
    "GlcNAc": "acgam_e", "Sucrose": "sucr_e", "Gluconate": "glcn_e",
    "lactic acid": "lac__L_e", "Fumarate": "fum_e", "Malic acid": "mal__L_e",
    "Pyruvate": "pyr_e", "Aspartate": "asp__L_e", "Glutamate": "glu__L_e",
    "Threonine": "thr__L_e", "Arginine": "arg__L_e", "Histidine": "his__L_e",
    "Lysine": "lys__L_e"
}

target_metabolite = "pyr_c"
results = []

for name, ext_id in carbon_sources.items():
    try:
        step_count = nx.shortest_path_length(G, source=ext_id, target=target_metabolite)
        path = nx.shortest_path(G, source=ext_id, target=target_metabolite)
        results.append({
            "Carbon_Source": name,
            "Steps_to_Pyruvate": step_count,
            "Path": " -> ".join(path)
        })
    except nx.NetworkXNoPath:
        results.append({
            "Carbon_Source": name,
            "Steps_to_Pyruvate": None,
            "Path": "No Path"
        })

df_results = pd.DataFrame(results).sort_values(by="Steps_to_Pyruvate")
display(df_results)

sp_tag = sanitize_for_filename(sp)
base = f"{sp_tag}"  # No index suffix since this is a single model
df_path = out_dir / f"{base}__pathway_distance_to_pyr.csv"
df_results.to_csv(df_path, index=False)

   Carbon_Source  Steps_to_Pyruvate                                                                                                         Path
27      Pyruvate                2.0                                                                                      pyr_e -> pyr_p -> pyr_c
1        Alanine                3.0                                                                    ala__L_e -> ala__L_p -> ala__L_c -> pyr_c
24   lactic acid                3.0                                                                    lac__L_e -> lac__L_p -> lac__L_c -> pyr_c
9         Serine                3.0                                                                    ser__L_e -> ser__L_p -> ser__L_c -> pyr_c
26    Malic acid                3.0                                                                    mal__L_e -> mal__L_p -> mal__L_c -> pyr_c
25      Fumarate                4.0                                                                 fum_e -> fum_p -> fum_c -> mal

In [ ]:
# Switch output directory to mg1655
out_dir = Path("/Volumes/Versatile/ECOR/mg1655/output_rev10/pathway_distance")
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Load the iML1515 model
# (download from the BIGG database or load from a local file)
sp = "iML1515"  # species name (also used in filenames)
model_path = "/Volumes/Versatile/ECOR/mg1655/iML1515.xml"
model = read_sbml_model(model_path)

# 2. Exclusion list (cofactors, shared metabolites, and quinones)
EXCLUDE_METABOLITES = {
    "atp_c", "adp_c", "amp_c", "pi_c", "ppi_c", 
    "nad_c", "nadh_c", "nadp_c", "nadph_c", 
    "h2o_c", "h_c", "h_p", "co2_c", "o2_c", "coa_c",
    "nh4_c", "so4_c", "cl_c", "na1_c", "k_c", "ca2_c", "mg2_c",
    "q8_c", "q8h2_c", "mqn8_c", "mqnh8_c", "2dmqn8_c", "2dmqnh8_c",
    "fad_c", "fadh2_c", "fe2_c", "fe3_c",
    "accoa_c", "micit_c", "actp_c"
}

# 3. Bypass reactions to knock out
EXCLUDE_REACTIONS = {"GLCDpp", "UGD"}

# 4. Build the directed graph
G = nx.DiGraph()

for reaction in model.reactions:
    if reaction.id in EXCLUDE_REACTIONS:
        continue

    is_pts = "pts" in reaction.id.lower() or (
        "pep_c" in [m.id for m in reaction.reactants] and "pyr_c" in [m.id for m in reaction.products]
    )

    reactants = [m.id for m in reaction.reactants if m.id not in EXCLUDE_METABOLITES]
    products = [m.id for m in reaction.products if m.id not in EXCLUDE_METABOLITES]

    if is_pts:
        products = [p for p in products if p != "pyr_c"]

    if reaction.upper_bound > 0:
        for r in reactants:
            for p in products:
                G.add_edge(r, p)
                
    if reaction.lower_bound < 0:
        for p in products:
            for r in reactants:
                G.add_edge(p, r)

# 5. Define the list of target carbon sources
carbon_sources = {
    "Succinate": "succ_e", "Alanine": "ala__L_e", "Proline": "pro__L_e",
    "Acetate": "ac_e", "Mannitol": "mnl_e", "Citrate": "cit_e",
    "Glucose": "glc__D_e", "Ribose": "rib__D_e", "Glycerol": "glyc_e",
    "Serine": "ser__L_e", "Sorbitol": "sbt__D_e", "Maltose": "malt_e",
    "Galactose": "gal_e", "Mannose": "man_e", "Trehalose": "tre_e",
    "Lactose": "lcts_e", "Melibiose": "melib_e", "Fructose": "fru_e",
    "Arabinose": "arab__L_e", "Rhamnose": "rmn_e", "Xylose": "xyl__D_e",
    "GlcNAc": "acgam_e", "Sucrose": "sucr_e", "Gluconate": "glcn_e",
    "lactic acid": "lac__L_e", "Fumarate": "fum_e", "Malic acid": "mal__L_e",
    "Pyruvate": "pyr_e", "Aspartate": "asp__L_e", "Glutamate": "glu__L_e",
    "Threonine": "thr__L_e", "Arginine": "arg__L_e", "Histidine": "his__L_e",
    "Lysine": "lys__L_e"
}

names = list(carbon_sources.keys())
n = len(names)

# 6. Compute the all-pairs directed shortest-path distance matrix
dist_matrix = np.full((n, n), np.nan)

for i, src_name in enumerate(names):
    src_id = carbon_sources[src_name]
    for j, tgt_name in enumerate(names):
        tgt_id = carbon_sources[tgt_name]
        if i == j:
            dist_matrix[i, j] = 0
        else:
            try:
                # Shortest number of steps from src to tgt
                dist_matrix[i, j] = nx.shortest_path_length(G, source=src_id, target=tgt_id)
            except nx.NetworkXNoPath:
                dist_matrix[i, j] = np.nan

# Convert the directed distance matrix to a DataFrame
df_dist_directed = pd.DataFrame(dist_matrix, index=names, columns=names)

# 7. Symmetrised distance (for structural similarity assessment)
# To capture the metabolic "closeness" of carbon sources A and B
# while accounting for directionality (one-way reactions),
# adopt min(dist(A→B), dist(B→A)).
sym_dist_matrix = np.full((n, n), np.nan)

for i in range(n):
    for j in range(n):
        d_ij = dist_matrix[i, j]
        d_ji = dist_matrix[j, i]
        
        # Use the minimum of the two directional distances (handles asymmetric cases)
        valid_dists = [d for d in [d_ij, d_ji] if not np.isnan(d)]
        if valid_dists:
            sym_dist_matrix[i, j] = min(valid_dists)
        else:
            sym_dist_matrix[i, j] = np.nan

df_dist_symmetric = pd.DataFrame(sym_dist_matrix, index=names, columns=names)

# 8. Build a flat data frame for per-pair similarity ranking
pair_list = []
for i in range(n):
    for j in range(i + 1, n):
        src = names[i]
        tgt = names[j]
        d_forward = df_dist_directed.loc[src, tgt]
        d_reverse = df_dist_directed.loc[tgt, src]
        min_dist = df_dist_symmetric.loc[src, tgt]
        
        pair_list.append({
            "Carbon_Source_A": src,
            "Carbon_Source_B": tgt,
            "Min_Distance": min_dist,
            "Dist_A_to_B": d_forward,
            "Dist_B_to_A": d_reverse
        })

df_pairs = pd.DataFrame(pair_list).sort_values(by="Min_Distance")

# Display results
display(df_pairs.head(20))

sp_tag = sanitize_for_filename(sp)
base = f"{sp_tag}"  # No index suffix since this is a single model
df_path = out_dir / f"{base}__pathway_distance_pairs.csv"
df_pairs.to_csv(df_path, index=False)

,Carbon_Source_A,Carbon_Source_B,Min_Distance,Dist_A_to_B,Dist_B_to_A
24,Succinate,Fumarate,3.0,3.0,3.0
25,Succinate,Malic acid,3.0,3.0,3.0
27,Succinate,Aspartate,3.0,3.0,3.0
332,Galactose,Lactose,3.0,NaN,3.0
190,Glucose,Trehalose,3.0,NaN,3.0
191,Glucose,Lactose,3.0,NaN,3.0
4,Succinate,Citrate,3.0,5.0,3.0
525,Fumarate,Malic acid,4.0,4.0,4.0
177,Citrate,Aspartate,4.0,4.0,6.0
534,Malic acid,Aspartate,4.0,4.0,4.0


<div class="alert-danger">
<font size="6">Miscs</font>
</div>

<div class="alert-success">
This is a green colored box
</div>

<div class="alert-danger">
This is a red colored box
</div>

<div class="alert-warning">
This is a yellow colored box
</div>

<div class="alert-info">
This is a blue colored box
</div>

In [ ]:
https://pmc.ncbi.nlm.nih.gov/articles/PMC10634833/#S9
https://pmc.ncbi.nlm.nih.gov/articles/PMC10173371/#S3
https://pmc.ncbi.nlm.nih.gov/articles/PMC7660451/#sec4